In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT = Path('BioReasoningChallenge/data')
LOCAL_DATA = PROJECT_ROOT / 'data'

DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_DATA
print(f'Using data directory: {DATA_DIR}')

if Path('/kaggle/input').exists():
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))


Using data directory: /Users/annaantipova/Desktop/bioreasoning/BioReasoningChallenge/data


In [3]:
SEEDS = [42, 43, 44]

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

prompt_candidates = [
    Path('/kaggle/input/datasets/nident/prompts/prompt.txt'),
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '04_combined_de_first_fewshot_strict.txt',
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '00_zero_shot_strict.txt',
]
default_prompt_path = next((path for path in prompt_candidates if path.exists()), None)
if default_prompt_path is None:
    raise FileNotFoundError('No prompt template found in Kaggle input or local prompt_strategies/track_a')

default_prompt = default_prompt_path.read_text(encoding='utf-8')

print(f'Using prompt template: {default_prompt_path}')
print(default_prompt[:1000])
print(train.columns.tolist())
print(train.head(2))
print(test.head(2))


Using prompt template: /Users/annaantipova/Desktop/bioreasoning/BioReasoningChallenge/prompt_strategies/track_a/04_combined_de_first_fewshot_strict.txt
### Instructions

### Role
You are an expert molecular biologist and computational genomics analyst. Your specialty is predicting gene-expression responses to CRISPRi perturbations in mouse bone marrow-derived macrophages.

### Task
For the current pair `{pert}` -> `{gene}`, predict the ternary Perturb-seq label:
A = knockdown of `{pert}` significantly up-regulates `{gene}`.
B = knockdown of `{pert}` significantly down-regulates `{gene}`.
C = knockdown of `{pert}` does not significantly affect `{gene}`.

Use this methodology:
1. Differential-expression check: decide whether `{gene}` is likely to pass the significant-effect threshold after `{pert}` knockdown.
2. If the significant-effect evidence is weak, choose C.
3. Direction check: if significant differential expression is likely, choose A or B based on regulatory direction.
4. Calibr

In [4]:
import os
from pprint import pprint
from openai import OpenAI

os.environ["OPENROUTER_API_KEY"] 
api_key = os.getenv('OPENROUTER_API_KEY')

if not api_key:
    print('OPENROUTER_API_KEY is not set; skipping live OpenRouter smoke test.')
else:
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key,
    )

    sample = train.iloc[0]
    prompt = default_prompt.format(
        pert=sample['pert'],
        gene=sample['gene'],
    )

    response = client.chat.completions.create(
        model='openai/gpt-oss-120b:free',
        messages=[{'role': 'user', 'content': prompt}],
        extra_body={'reasoning': {'enabled': True}},
    )

    pprint(response.choices[0].message.content)


'<answer>C</answer>'


In [ ]:
from src.prompt_tester import PromptTester


print(train.head())

tester = PromptTester(
    client=client,
    model='openai/gpt-oss-120b:free',
    max_samples=5,  # Or None for all
)

# Test both prompts
prompts_to_test = [
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '00_zero_shot_strict.txt',
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '04_combined_de_first_fewshot_strict.txt',
]

for prompt_path in prompts_to_test:
    if prompt_path.exists():
        result = tester.test_prompt(prompt_path, train, label_column='label')
        tester.results[prompt_path.name] = result
    else:
        print(f"⚠️ Prompt not found: {prompt_path}")

# 🏆 Comparison
print("\n" + "="*70)
print("🏆 PROMPT COMPARISON")
print("="*70)
comparison = tester.compare_prompts()
print(comparison.to_string(index=False))

# 📊 Detailed results for each prompt
for prompt_name in comparison['Prompt']:
    tester.print_detailed_results(prompt_name)

# Save everything
tester.save_analysis(Path('prompt_analysis'))